# Silver to Gold Transformation
**Project:** NYC TLC Azure Data Platform  
**Layer:** Gold  
**Description:** Reads clean Silver Delta tables, builds aggregated Gold tables for Power BI reporting  
**Author:** Saikumar 
**Created:** 2026

In [0]:
from pyspark.sql.functions import (
    col, lit, when, sum, count, avg,
    year, month, hour, dayofweek,
    round, broadcast, to_date, date_format
)

#Storage configuration
STORAGE_ACCOUNT = "nycrawdatazone"
CONTAINER       = "nyc-tlc"
ADLS_ENDPOINT   = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

#Layer paths
SILVER_PATH = f"{ADLS_ENDPOINT}/silver"
GOLD_PATH   = f"{ADLS_ENDPOINT}/gold"

#Spark optimizations
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print(f"Silver path : {SILVER_PATH}")
print(f"Gold path   : {GOLD_PATH}")

In [0]:
# Reading Silver tables
yellow_df = spark.read.format("delta").load(f"{SILVER_PATH}/yellow")
green_df  = spark.read.format("delta").load(f"{SILVER_PATH}/green")
fhv_df    = spark.read.format("delta").load(f"{SILVER_PATH}/fhv")
fhvhv_df  = spark.read.format("delta").load(f"{SILVER_PATH}/fhvhv")

print(f"Yellow : {yellow_df.count():>15,} rows")
print(f"Green  : {green_df.count():>15,} rows")
print(f"FHV    : {fhv_df.count():>15,} rows")
print(f"FHVHV  : {fhvhv_df.count():>15,} rows")

#Gold Table 1 - Monthly Revenue

In [0]:
def build_monthly_revenue():
     
    #Prepare Yellow
    yellow = yellow_df.select(
    year("tpep_pickup_datetime").alias("year"),
    month("tpep_pickup_datetime").alias("month"),
    "taxi_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "cbd_congestion_fee",
    "congestion_surcharge",
    "airport_fee",
    "trip_distance",
    "trip_duration_mins",
    "passenger_count"
)

    # Green
    green = green_df.select(
    year("lpep_pickup_datetime").alias("year"),
    month("lpep_pickup_datetime").alias("month"),
    "taxi_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "cbd_congestion_fee",
    "congestion_surcharge",
    lit(0.0).alias("airport_fee"),
    "trip_distance",
    "trip_duration_mins",
    "passenger_count"
    )

    # FHVHV
    fhvhv = fhvhv_df.select(
    year("pickup_datetime").alias("year"),
    month("pickup_datetime").alias("month"),
    "taxi_type",
    col("base_passenger_fare").alias("fare_amount"),
    col("tips").alias("tip_amount"),
    col("tolls").alias("tolls_amount"),
    (col("base_passenger_fare") +
     col("tips") +
     col("tolls") +
     col("congestion_surcharge") +
     col("airport_fee") +
     col("cbd_congestion_fee")).alias("total_amount"),
    "cbd_congestion_fee",
    "congestion_surcharge",
    "airport_fee",
    col("trip_miles").alias("trip_distance"),
    "trip_duration_mins",
    lit(1).alias("passenger_count")
    )
    #Union Yellow + Green + FHVHV
    combined = yellow.union(green).union(fhvhv)

    #Aggregate
    monthly_revenue = combined.groupBy(
        "year",
        "month",
        "taxi_type"
    ).agg(
        count("*").alias("total_trips"),
        round(sum("fare_amount"), 2).alias("total_fare"),
        round(sum("tip_amount"), 2).alias("total_tips"),
        round(sum("tolls_amount"), 2).alias("total_tolls"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(sum("cbd_congestion_fee"), 2).alias("total_congestion_fee"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("tip_amount"), 2).alias("avg_tip"),
        round(avg("total_amount"), 2).alias("avg_total"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration_mins"),
        round(avg("passenger_count"), 2).alias("avg_passengers")
    ).orderBy("year", "month", "taxi_type")

    print(f"Rows : {monthly_revenue.count():,}")
    display(monthly_revenue.show(10))

    #Write to Gold
    monthly_revenue.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{GOLD_PATH}/monthly_revenue")

    print("monthly_revenue written to gold layer")

build_monthly_revenue()

#Gold Table 2 - CBD Impact

In [0]:
def build_cbd_impact():
    #Yellow
    yellow = yellow_df.select(
        year("tpep_pickup_datetime").alias("year"),
        month("tpep_pickup_datetime").alias("month"),
        "taxi_type",
        "cbd_congestion_fee",
        "fare_amount",
        "total_amount",
        "pickup_borough",
        "pickup_zone"
    )
    #Green
    green = green_df.select(
        year("lpep_pickup_datetime").alias("year"),
        month("lpep_pickup_datetime").alias("month"),
        "taxi_type",
        "cbd_congestion_fee",
        "fare_amount",
        "total_amount",
        "pickup_borough",
        "pickup_zone"
    )
    #FHVHV
    fhvhv = fhvhv_df.select(
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"),
        "taxi_type",
        "cbd_congestion_fee",
        col("base_passenger_fare").alias("fare_amount"),
        (col("base_passenger_fare") +
         col("tips") +
         col("tolls") +
         col("congestion_surcharge") +
         col("airport_fee") +
         col("cbd_congestion_fee")).alias("total_amount"),
        "pickup_borough",
        "pickup_zone"
    )

    #Union all 3
    combined = yellow.union(green).union(fhvhv)

    #Add CBD flag
    combined = combined.withColumn("is_cbd_trip",
        when(col("cbd_congestion_fee") > 0, 1).otherwise(0)
    )

    #Aggregate
    cbd_impact = combined.groupBy(
        "year",
        "month",
        "taxi_type"
    ).agg(
        count("*").alias("total_trips"),
        sum("is_cbd_trip").alias("cbd_trips"),
        (count("*") - sum("is_cbd_trip")).alias("non_cbd_trips"),
        round(
            sum("is_cbd_trip") / count("*") * 100, 2
        ).alias("cbd_trip_rate_pct"),
        round(sum("cbd_congestion_fee"), 2).alias("total_congestion_revenue"),
        round(avg("cbd_congestion_fee"), 2).alias("avg_congestion_fee"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("total_amount"), 2).alias("avg_total")
    ).orderBy("year", "month", "taxi_type")

    print(f"Rows : {cbd_impact.count():,}")
    display(cbd_impact.limit(40))

    #Write to Gold
    cbd_impact.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{GOLD_PATH}/cbd_impact")

    print("cbd_impact written to gold layer")

build_cbd_impact()

#Gold Table 3 - Zone Demand (pickup)

In [0]:
def build_zone_demand():
    #Prepare Yellow
    yellow = yellow_df.select(
        year("tpep_pickup_datetime").alias("year"),
        month("tpep_pickup_datetime").alias("month"),
        "taxi_type",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
        "fare_amount",
        "total_amount",
        "trip_distance",
        "trip_duration_mins",
        "cbd_congestion_fee"
    )

    #Prepare Green
    green = green_df.select(
        year("lpep_pickup_datetime").alias("year"),
        month("lpep_pickup_datetime").alias("month"),
        "taxi_type",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
        "fare_amount",
        "total_amount",
        "trip_distance",
        "trip_duration_mins",
        "cbd_congestion_fee"
    )

    #Prepare FHV
    fhv = fhv_df.select(
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"),
        "taxi_type",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
        lit(0.0).alias("fare_amount"),
        lit(0.0).alias("total_amount"),
        lit(0.0).alias("trip_distance"),
        "trip_duration_mins",
        lit(0.0).alias("cbd_congestion_fee")
    )

    #Prepare FHVHV
    fhvhv = fhvhv_df.select(
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"),
        "taxi_type",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
        col("base_passenger_fare").alias("fare_amount"),
        (col("base_passenger_fare") +
         col("tips") +
         col("tolls") +
         col("congestion_surcharge") +
         col("airport_fee") +
         col("cbd_congestion_fee")).alias("total_amount"),
        col("trip_miles").alias("trip_distance"),
        "trip_duration_mins",
        "cbd_congestion_fee"
    )

    #Union all 4
    combined = yellow.union(green).union(fhv).union(fhvhv)

    #Aggregate by pickup zone
    zone_demand = combined.groupBy(
        "year",
        "month",
        "taxi_type",
        "pickup_borough",
        "pickup_zone"
    ).agg(
        count("*").alias("total_pickups"),
        round(sum("fare_amount"), 2).alias("total_fare"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration_mins"),
        round(sum("cbd_congestion_fee"), 2).alias("total_congestion_fee")
    ).orderBy("year", "month", "taxi_type", "total_pickups")

    print(f"Rows : {zone_demand.count():,}")
    zone_demand.show(10)

    #Write to Gold
    zone_demand.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{GOLD_PATH}/zone_demand")

    print("zone_demand written to gold layer")

build_zone_demand()

#Gold Table - DropOff Demand

In [0]:
def build_dropoff_demand():
    
    #Prepare Yellow
    yellow = yellow_df.select(
        year("tpep_pickup_datetime").alias("year"),
        month("tpep_pickup_datetime").alias("month"),
        "taxi_type",
        "dropoff_borough",
        "dropoff_zone",
        "fare_amount",
        "total_amount",
        "trip_distance",
        "trip_duration_mins"
    )

    #Prepare Green
    green = green_df.select(
        year("lpep_pickup_datetime").alias("year"),
        month("lpep_pickup_datetime").alias("month"),
        "taxi_type",
        "dropoff_borough",
        "dropoff_zone",
        "fare_amount",
        "total_amount",
        "trip_distance",
        "trip_duration_mins"
    )

    #Prepare FHV
    fhv = fhv_df.select(
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"),
        "taxi_type",
        "dropoff_borough",
        "dropoff_zone",
        lit(0.0).alias("fare_amount"),
        lit(0.0).alias("total_amount"),
        lit(0.0).alias("trip_distance"),
        "trip_duration_mins"
    )

    #Prepare FHVHV
    fhvhv = fhvhv_df.select(
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"),
        "taxi_type",
        "dropoff_borough",
        "dropoff_zone",
        col("base_passenger_fare").alias("fare_amount"),
        (col("base_passenger_fare") +
         col("tips") +
         col("tolls") +
         col("congestion_surcharge") +
         col("airport_fee") +
         col("cbd_congestion_fee")).alias("total_amount"),
        col("trip_miles").alias("trip_distance"),
        "trip_duration_mins"
    )

    # Union all 4
    combined = yellow.union(green).union(fhv).union(fhvhv)

    # Aggregate by dropoff zone
    dropoff_demand = combined.groupBy(
        "year",
        "month",
        "taxi_type",
        "dropoff_borough",
        "dropoff_zone"
    ).agg(
        count("*").alias("total_dropoffs"),
        round(sum("fare_amount"), 2).alias("total_fare"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration_mins")
    ).orderBy("year", "month", "taxi_type", "total_dropoffs")

    print(f"Rows : {dropoff_demand.count():,}")
    dropoff_demand.show(10)

    # Write to Gold
    dropoff_demand.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{GOLD_PATH}/dropoff_demand")

    print("dropoff_demand written to gold layer")

build_dropoff_demand()

#Gold Tale 4 - Hourly Patterns

In [0]:
def build_hourly_patterns():
    
    #Yellow
    yellow = yellow_df.select(
        year("tpep_pickup_datetime").alias("year"),
        month("tpep_pickup_datetime").alias("month"),
        hour("tpep_pickup_datetime").alias("hour"),
        dayofweek("tpep_pickup_datetime").alias("day_of_week"),
        "taxi_type",
        "fare_amount",
        "trip_duration_mins",
        "trip_distance",
        "cbd_congestion_fee"
    )

    #Green
    green = green_df.select(
        year("lpep_pickup_datetime").alias("year"),
        month("lpep_pickup_datetime").alias("month"),
        hour("lpep_pickup_datetime").alias("hour"),
        dayofweek("lpep_pickup_datetime").alias("day_of_week"),
        "taxi_type",
        "fare_amount",
        "trip_duration_mins",
        "trip_distance",
        "cbd_congestion_fee"
    )

    #FHV
    fhv = fhv_df.select(
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"),
        hour("pickup_datetime").alias("hour"),
        dayofweek("pickup_datetime").alias("day_of_week"),
        "taxi_type",
        lit(0.0).alias("fare_amount"),
        "trip_duration_mins",
        lit(0.0).alias("trip_distance"),
        lit(0.0).alias("cbd_congestion_fee")
    )

    #FHVHV
    fhvhv = fhvhv_df.select(
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"),
        hour("pickup_datetime").alias("hour"),
        dayofweek("pickup_datetime").alias("day_of_week"),
        "taxi_type",
        col("base_passenger_fare").alias("fare_amount"),
        "trip_duration_mins",
        col("trip_miles").alias("trip_distance"),
        "cbd_congestion_fee"
    )

    #Union all 4
    combined = yellow.union(green).union(fhv).union(fhvhv)

    #Adding day type
    # day_of_week: 1=Sunday, 2=Monday ... 7=Saturday
    combined = combined.withColumn("day_type",
        when(
            col("day_of_week").isin(1, 7), "weekend"
        ).otherwise("weekday")
    )

    #Aggregate
    hourly_patterns = combined.groupBy(
        "year",
        "month",
        "hour",
        "day_type",
        "taxi_type"
    ).agg(
        count("*").alias("total_trips"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration_mins"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(sum("cbd_congestion_fee"), 2).alias("total_congestion_fee"),
        round(avg("cbd_congestion_fee"), 2).alias("avg_congestion_fee")
    ).orderBy("year", "month", "hour", "day_type", "taxi_type")

    print(f"Rows : {hourly_patterns.count():,}")
    hourly_patterns.show(10)

    #Write to Gold
    hourly_patterns.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{GOLD_PATH}/hourly_patterns")

    print("hourly_patterns written to gold layer")

build_hourly_patterns()

In [0]:
# Verifying all Gold tables
print("Verifying Gold layer...\n")

gold_tables = [
    "monthly_revenue",
    "cbd_impact",
    "zone_demand",
    "hourly_patterns",
    "dropoff_demand"
]

for table in gold_tables:
    try:
        df = spark.read.format("delta").load(f"{GOLD_PATH}/{table}")
        print(f"{table:20} : {df.count():>8,} rows | {len(df.columns)} columns")
    except Exception as e:
        print(f"{table:20} : {str(e)[:100]}")

print("\n Gold layer complete!")